# Parte 01

## Importação das bibliotecas

In [1]:
print("Teste de commit executado com sucesso!")

Teste de commit executado com sucesso!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

## Preparação da carga dataframe

In [ ]:
# Lista com o dicionario de dados
colunas = [
    "ID_CLIENTE","TIPO_FUNCIONARIO","DIA_PAGAMENTO",
    "TIPO_ENVIO_APLICACAO","QUANT_CARTOES_ADICIONAIS",
    "TIPO_ENDERECO_POSTAL","SEXO","ESTADO_CIVIL",
    "QUANT_DEPENDENTES","NIVEL_EDUCACIONAL",
    "ESTADO_NASCIMENTO","CIDADE_NASCIMENTO",
    "NACIONALIDADE","ESTADO_RESIDENCIAL",
    "CIDADE_RESIDENCIAL","BAIRRO_RESIDENCIAL",
    "FLAG_TELEFONE_RESIDENCIAL",
    "CODIGO_AREA_TELEFONE_RESIDENCIAL",
    "TIPO_RESIDENCIA","MESES_RESIDENCIA",
    "FLAG_TELEFONE_MOVEL","FLAG_EMAIL",
    "RENDA_PESSOAL_MENSAL","OUTRAS_RENDAS",
    "FLAG_VISA","FLAG_MASTERCARD","FLAG_DINERS",
    "FLAG_AMERICAN_EXPRESS","FLAG_OUTROS_CARTOES",
    "QUANT_CONTAS_BANCARIAS",
    "QUANT_CONTAS_BANCARIAS_ESPECIAIS",
    "VALOR_PATRIMONIO_PESSOAL","QUANT_CARROS",
    "EMPRESA","ESTADO_PROFISSIONAL",
    "CIDADE_PROFISSIONAL","BAIRRO_PROFISSIONAL",
    "FLAG_TELEFONE_PROFISSIONAL",
    "CODIGO_AREA_TELEFONE_PROFISSIONAL",
    "MESES_NO_TRABALHO","CODIGO_PROFISSAO",
    "TIPO_OCUPACAO","CODIGO_PROFISSAO_CONJUGE",
    "NIVEL_EDUCACIONAL_CONJUGE",
    "FLAG_DOCUMENTO_RESIDENCIAL","FLAG_RG",
    "FLAG_CPF","FLAG_COMPROVANTE_RENDA",
    "PRODUTO","FLAG_REGISTRO_ACSP",
    "IDADE","CEP_RESIDENCIAL_3",
    "CEP_PROFISSIONAL_3",
    "ROTULO_ALVO_MAU"
]

df = pd.read_csv("https://raw.githubusercontent.com/diogenesjusto/FIAP/master/dados/credit.csv", encoding='unicode_escape', sep='\t', header=None,names=colunas)

df.head()

## Visualização Inicial

In [ ]:
print("Linhas:", df.shape[0])
print("Colunas:", df.shape[1])



df.info()

## Valores Ausentes

In [ ]:
missing = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
      .to_frame("Percentual")
)

missing.head(20)

## Convert colunas no data frame para numerico

In [ ]:
for coluna in df.columns:
    try:
        df[coluna] = pd.to_numeric(df[coluna])
    except:
        pass

## Estatistica Descritiva

In [ ]:
df.describe().T

## Analise da distribuição da variavel alvo

In [ ]:
df["ROTULO_ALVO_MAU"].value_counts()

In [ ]:
df["ROTULO_ALVO_MAU"].value_counts()

df["ROTULO_ALVO_MAU"].value_counts().plot(
    kind="bar",
    figsize=(6,4),
    title="Distribuição da Classe"
)

plt.show()

## Proporção de clientes bom pagadores maior qwe de maus pagadores

Análise da Renda por classe

In [ ]:
df.groupby("ROTULO_ALVO_MAU")[
    "RENDA_PESSOAL_MENSAL"
].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="RENDA_PESSOAL_MENSAL"
)

plt.title("Renda por Classe")
plt.show()

Conclusão: A variável RENDA_PESSOAL_MENSAL está muito concentrada em valores baixos para as duas classes (ROTULO_ALVO_MAU = 0 e ROTULO_ALVO_MAU = 1), mas existem muitos outliers altos que distorcem a escala do gráfico.
A renda mensal, isoladamente, não parece ser uma variável suficiente para distinguir claramente clientes bons de clientes maus.
Uma possibilidade para gerar um gráfico mais adequado a como a variavel esta (sem recorrer a nenhuma tecnica de normalização dos dados) e adotar a escala logaritmica ou restringir usando um limite no eixo Y

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="RENDA_PESSOAL_MENSAL"
)

plt.ylim(0, df["RENDA_PESSOAL_MENSAL"].quantile(0.95))
plt.title("Renda por Classe sem outliers extremos")
plt.show()

## Análise da Idade do cliente

In [ ]:
df.groupby("ROTULO_ALVO_MAU")[
    "IDADE"
].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="IDADE"
)

plt.title("Idade por Classe")
plt.show()

## Análise e Tratamento varieis categoricas

In [ ]:
print( df["ESTADO_CIVIL"].unique())
print( df["EMPRESA"].unique())
print( df["PRODUTO"].unique())
print( df["SEXO"].unique())
print( df["TIPO_ENVIO_APLICACAO"].unique())




In [ ]:
#ajustes nas colunas sexo e tipo envio aplicação
df.loc[df["TIPO_ENVIO_APLICACAO"] == 'Web', "TIPO_ENVIO_APLICACAO"] = '1'
df.loc[df["TIPO_ENVIO_APLICACAO"] == 'Carga', "TIPO_ENVIO_APLICACAO"] = '2'



In [ ]:
df.loc[df["SEXO"] == 'M', "SEXO"] = '1'
df.loc[df["SEXO"] == 'F', "SEXO"] = '0'
df.loc[df["SEXO"] == 'N', "SEXO"] = '-1'
df.loc[df["SEXO"] == ' ', "SEXO"] = '-1'
# -1 SEXO NÃO INFORMADO

In [ ]:
variaveis_categoricas = [
    "ESTADO_CIVIL",
    "EMPRESA",
    "PRODUTO",
    "TIPO_ENVIO_APLICACAO",
    "SEXO"
]

for var in variaveis_categoricas:

    tabela = (
        df.groupby(var)["ROTULO_ALVO_MAU"]
          .agg(
              quantidade="count",
              maus="sum",
              taxa_mau="mean"
          )
          .sort_values(
              "taxa_mau",
              ascending=False
          )
    )

    print(f"\n===== {var} =====")
    display(tabela)

## Calculo da correlação variavel alvo (ROTULO_ALVO_MAU)

In [ ]:
numericas = df.select_dtypes(include=np.number)

correlacoes = (
    numericas.corr(method="spearman")
              ["ROTULO_ALVO_MAU"]
              .sort_values(
                  key=lambda x: abs(x),
                  ascending=False
              )
)

print(correlacoes)

In [ ]:
correlacoes.head(20).sort_values().plot(
    kind="barh",
    figsize=(10,8)
)

plt.title(
    "Correlação com ROTULO_ALVO_MAU"
)

plt.show()

In [ ]:
# Visualizando a matriz em um mapa de calor (heatmap)

data = correlacoes.head(20).sort_values().to_frame()
plt.figure(figsize=(25, 20))
sns.heatmap(data, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação')
plt.show()


# Parte 02

## Como as features sexo, estado civil, quantidade de dependentes e nível educacional está relacionada com a variável meta?[texto do link](https://)

In [ ]:
features = [

    "SEXO",
    "ESTADO_CIVIL",
    "QUANT_DEPENDENTES",
    "NIVEL_EDUCACIONAL"

]

In [ ]:


alvo = "ROTULO_ALVO_MAU"



for col in features:

    print(f"\n==============================")

    print(f"Feature: {col}")

    print(f"==============================")



    tabela = (

        df.groupby(col)[alvo]

        .agg(

            total_clientes="count",

            total_maus="sum",

            taxa_mau="mean"

        )

        .reset_index()

        .sort_values("taxa_mau", ascending=False)

    )



    tabela["taxa_mau_%"] = tabela["taxa_mau"] * 100



    display(tabela)

In [ ]:


for col in features:

    resumo = (

        df.groupby(col)[alvo]

        .mean()

        .reset_index()

        .sort_values(alvo, ascending=False)

    )



    plt.figure(figsize=(10, 4))

    sns.barplot(data=resumo, x=col, y=alvo)

    plt.title(f"Taxa média de ROTULO_ALVO_MAU por {col}")

    plt.ylabel("Taxa de mau pagador")

    plt.xlabel(col)

    plt.xticks(rotation=45)

    plt.show()



 **Afirmações com base nos dados:**
 - O dataset usado tem 50.000 registros, 54 colunas, alvo ROTULO_ALVO_MAU, com 36.959 bons pagadores — 73,92% — e 13.041 maus pagadores — 26,08%.
  + Sobre a feature sexo, homens aparecem com uma taxa de mau pagador um pouco maior que mulheres, mas a diferença é de aproximadamente 2 pontos percentuais, então eu classificaria essa variável como baixo poder discriminante isoladamente.
  + No caso da feature Estado_Civil, O código 1 apresenta uma proporção maior de maus pagadores, enquanto o código 4 apresenta uma proporção menor. A fonte analisada sugere que o código 1 pode representar “solteiro?” e o código 4 pode representar “casado?”, mas isto sem um dicionário de dados oficial não passa de uma especulação do que o código poderia ser. O ESTADO_CIVIL parece ter uma relação mais relevante com o alvo do que SEXO.
  + Sobre a feature Nivel_Educacional como todos os registros têm o mesmo valor, não existe variação para comparar bons e maus pagadores. Portanto, NIVEL_EDUCACIONAL não tem poder explicativo nesse dataset específico e deve ser tratada como variável sem utilidade preditiva


**As features SEXO, ESTADO_CIVIL e QUANT_DEPENDENTES apresentam alguma associação com a variável alvo ROTULO_ALVO_MAU, porém com intensidades diferentes. Para identificar a relação com a variável meta (ROTULO_ALVO_MAU) vamos lançar mão de um teste qui- quadrado (para medir associação estatistica) e um Cramer's V ( verificar a força desta associação)**

In [ ]:

from scipy.stats import chi2_contingency

def cramers_v(tabela):

  chi2 = chi2_contingency(tabela)[0]

  n = tabela.sum().sum()

  r, k = tabela.shape

  return np.sqrt((chi2 / n) / min(r - 1, k - 1))

resultado = []



for feature in features:

  crosstab = pd.crosstab(df[feature], df['ROTULO_ALVO_MAU'])

  chi2, p_value, _, _ = chi2_contingency(crosstab)

  cv = cramers_v(crosstab)
  taxa = (df.groupby(feature)['ROTULO_ALVO_MAU'].mean().reset_index().sort_values('ROTULO_ALVO_MAU', ascending=False))
  print(f"\n{'='*60}")
  print(feature)
  print(taxa)

  resultado.append({"feature": feature,"p_value": p_value,"cramers_v": cv})


resultado_df = pd.DataFrame(resultado)
resultado_df.sort_values('cramers_v', ascending=False)

**Base para Interpretação**
- p-value < 0,05 ---> Existe associação estatistica
- Cramer's V < 0,10 ---> Associação muito fraca
- Cramer's entre 0,10 a 0,30 ---> Associação muito fraca/moderada
- Cramer's entre 0,30 a 0,50 ---> Associação moderada
- Cramer's V > 0,50 ---> Associação forte


###**Conclusão Principal**
Nenhuma das quatro variáveis apresenta uma relação forte com a variável meta ROTULO_ALVO_MAU.

Mesmo quando o p-value é muito baixo, como em ESTADO_CIVIL e SEXO, o Cramér's V é muito pequeno. Isso significa que existe alguma diferença estatística entre os grupos, mas essa diferença tem baixo poder explicativo


##Plote um gráfico que mostre a distribuição de bons e maus pagadores por estado. Escolha o gráfico que achar mais conveniente

In [ ]:
# Criar uma cópia para não alterar o dataframe original

df_plot = df.copy()

# Criar rótulo para facilitar a leitura

df_plot["STATUS_PAGADOR"] = df_plot["ROTULO_ALVO_MAU"].map({

    0: "Bom pagador",

    1: "Mau pagador"

})

# Tabela de contagem por estado e status

tabela_estado = pd.crosstab(

    df_plot["ESTADO_RESIDENCIAL"],

    df_plot["STATUS_PAGADOR"]

)

# Ordenar pelo total de clientes por estado

tabela_estado["TOTAL"] = tabela_estado.sum(axis=1)

tabela_estado = tabela_estado.sort_values("TOTAL", ascending=False)

tabela_estado = tabela_estado.drop(columns="TOTAL")

# Plot

plt.figure(figsize=(14, 6))



tabela_estado.plot(

    kind="bar",

    stacked=True,

    figsize=(19, 10)

)



plt.title("Distribuição de bons e maus pagadores por estado residencial")

plt.xlabel("Estado residencial")

plt.ylabel("Quantidade de clientes")

plt.xticks(rotation=45)

plt.legend(title="Status do pagador")

plt.tight_layout()

plt.show()

